In [1]:
import torch
import torch.optim as optim

import json
import os
from tqdm import tqdm
import itertools
import logging

import network
import utils
import train

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


# Data Loading

In [4]:
medoid_loader = utils.get_medoid_loaders(64)
m2s_loader = utils.get_m2s_loader(128)

medoid_config_file = 'Config/medoid_config.json'
with open(medoid_config_file, 'r') as file:
    medoid_config = json.load(file)

m2s_config_file = 'Config/m2s_config.json'
with open(m2s_config_file, 'r') as file:
    m2s_config = json.load(file)

with open('Config/losses.json', 'r') as f:
    losses = json.load(f)

medoid_best_loss = losses['medoid_best_loss']
m2s_best_loss = losses['m2s_best_loss']

print('medoid_best_loss:', medoid_best_loss)
print('m2s_best_loss:', m2s_best_loss)

print('medoid_config:', medoid_config)
print('m2s_config:', m2s_config)

medoid_best_loss: 7.1898831649865516
m2s_best_loss: 100
medoid_config: {'encoder_hidden': 256, 'encoder_lstm_size': 64, 'encoder_lstm_layers': 2, 'latent_dim': 64, 'decoder_hidden_size': 1024, 'embed_size': 4, 'time_size': 128, 'decoder_lstm_size': 8, 'decoder_lstm_layers': 4}
m2s_config: {'encoder_hidden': 256, 'encoder_lstm_size': 8, 'encoder_lstm_layers': 2, 'latent_dim': 512, 'decoder_hidden': 512, 'embed_size_cluster': 2, 'embed_size_spike': 1, 'decoder_layers': 3, 'decoder_lstm_size': 32, 'decoder_lstm_dropout': 0.2, 'time_size': 32, 'statistics_size': 64, 'gradient_size': 128, 'fc_dropout': 0.2}


# Training the Medoid VAE

In [3]:
model_medoid = network.m_VAE(**medoid_config).to(device)

optimizer = optim.Adam(model_medoid.parameters(), lr=1e-4)

name = 'test_s'
loss, model = train.train_medoid_vae(model_medoid, 1000, medoid_loader, optimizer, device, name)

if loss < medoid_best_loss:
    print('New best loss:', loss)
    torch.save(model, f'../../Models/medoid_vae_best.pt')
    losses['medoid_best_loss'] = loss
    with open('Config/losses.json', 'w') as f:
        json.dump(losses, f, indent=4)
else:
    print('No improvement')
    torch.save(model, f'../../Models/medoid_vae_{name}.pt')

Training Progress:  51%|█████████▏        | 509/1000 [2:33:26<2:28:01, 18.09s/it, current loss=7.21]

Early stopping
Finished training, best loss: 7.1899
New best loss: 7.1898831649865516


# Training the Medoid-to-Spike VAE

In [5]:
model_medoid = network.m_VAE(**medoid_config).to(device)
model_medoid.load_state_dict(torch.load('../../Models/medoid_vae_best.pt'))
 
model_m2s = network.m2s_VAE(**m2s_config).to(device)
# model_r2s.init_weights()
optimizer = optim.Adam(model_m2s.parameters(), lr=3e-5)
 
best_loss, best_model = train.train_m2s_vae(model_m2s, model_medoid, 1000, m2s_loader, optimizer, device, 'test')

if best_loss < m2s_best_loss:
    print('New best loss:', best_loss)
    
    losses['m2s_best_loss'] = best_loss
    with open('Config/losses.json', 'w') as f:
        json.dump(losses, f, indent=4)

    torch.save(best_model, f'../../Models/m2s_vae_best.pt') 
else:
    print('No improvement')
    torch.save(best_model, f'../../Models/m2s_vae_test.pt')

Training Progress:   0%|                                                   | 0/1000 [04:18<?, ?it/s]


KeyboardInterrupt: 